# Lab 2.3: Error Propagation in Multi-Agent Systems

**What you'll build:** A multi-agent research pipeline with structured error propagation — and learn why silent suppression is worse than crashing.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way — silent error suppression produces wrong answers without warning | 5 min |
| 2 | The Right Way — structured error context enables intelligent recovery | 5 min |
| 3 | Your Turn — build error propagation for a data pipeline | 10 min |
| 4 | Stress Test — verify your error handling across failure modes | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You have a multi-agent research system with three subagents:
1. **Database agent** — queries internal databases
2. **Web search agent** — searches the internet
3. **File reader agent** — reads local documents

A **coordinator agent** sends queries to all three, collects results, and synthesizes a final answer.

The critical question: what happens when one subagent fails? The answer determines whether your system produces wrong answers silently or recovers intelligently.

In [ ]:
# Simulated subagent results — some succeed, some fail
def simulate_database_agent(query, fail_mode=None):
    """Simulate a database query that might fail."""
    if fail_mode == "timeout":
        # ANTI-PATTERN: silent suppression — returns empty as success
        return {"results": [], "status": "success"}
    elif fail_mode == "timeout_structured":
        # CORRECT: structured error propagation
        return {
            "results": [],
            "status": "error",
            "error_type": "connection_timeout",
            "attempted_query": query,
            "partial_results": None,
            "alternatives": ["retry_with_backoff", "use_cached_data"]
        }
    else:
        return {
            "results": [
                {"company": "Acme Corp", "revenue": "$4.2B", "year": 2024},
                {"company": "Acme Corp", "employees": 15000, "year": 2024}
            ],
            "status": "success"
        }

def simulate_web_agent(query, fail_mode=None):
    """Simulate a web search that might fail."""
    if fail_mode == "rate_limited":
        return {"results": [], "status": "success"}  # Silent suppression
    elif fail_mode == "rate_limited_structured":
        return {
            "results": [],
            "status": "error",
            "error_type": "rate_limited",
            "attempted_query": query,
            "partial_results": None,
            "alternatives": ["retry_after_60s", "use_alternative_search"]
        }
    else:
        return {
            "results": [
                {"title": "Acme Corp Q4 Earnings", "snippet": "Revenue reached $4.2B..."},
                {"title": "Acme Expansion Plans", "snippet": "Opening 3 new offices..."}
            ],
            "status": "success"
        }

def simulate_file_agent(query, fail_mode=None):
    """Simulate file reading that might fail."""
    if fail_mode == "permission_denied":
        return {"results": [], "status": "success"}  # Silent suppression
    elif fail_mode == "permission_denied_structured":
        return {
            "results": [],
            "status": "error",
            "error_type": "permission_denied",
            "attempted_query": query,
            "partial_results": None,
            "alternatives": ["request_access", "use_public_filing"]
        }
    else:
        return {
            "results": [
                {"file": "annual_report.pdf", "content": "Acme Corp reported record revenue..."}
            ],
            "status": "success"
        }

print("Simulated agents loaded.")
print("Each agent can succeed or fail in different modes.")

---

## Phase 1: The Wrong Approach

Silent suppression: when agents fail, they return empty results as if everything is fine. The coordinator synthesizes results without knowing data is missing.

In [ ]:
# Scenario: database times out, web is rate limited, only files succeed
query = "What is Acme Corp's current revenue and expansion plans?"

db_result = simulate_database_agent(query, fail_mode="timeout")
web_result = simulate_web_agent(query, fail_mode="rate_limited")
file_result = simulate_file_agent(query)  # succeeds

# The coordinator sees: 2 empty results + 1 real result
# But it thinks all queries succeeded!
all_results = {
    "database": db_result,
    "web_search": web_result,
    "file_reader": file_result
}

coordinator_prompt = f"""You are a research coordinator. You sent a query to three data sources.
Here are the results:

{json.dumps(all_results, indent=2)}

Query: {query}

Synthesize a comprehensive answer based on all available data.
Rate your confidence in the completeness of this answer (high/medium/low).

Respond as JSON: {{"answer": "...", "confidence": "...", "sources_used": [...]}}"""

response = client.messages.create(
    model=MODEL,
    max_tokens=500,
    messages=[{"role": "user", "content": coordinator_prompt}]
)

print("=== SILENT SUPPRESSION: Coordinator's Answer ===")
print(response.content[0].text)
print()
print("THE PROBLEM:")
print("  - Database timed out (coordinator doesn't know)")
print("  - Web search was rate-limited (coordinator doesn't know)")
print("  - Only file reader succeeded")
print("  - Coordinator may report HIGH confidence based on incomplete data")

### Why is this dangerous?

- The coordinator **cannot distinguish** "database had no records" from "database was unreachable."
- It synthesizes a confident-sounding answer from **1 out of 3 sources** without any warning.
- The user has **no way to know** that two data sources were not actually consulted.
- This is **worse than crashing** — a crash at least tells you something went wrong.

---

## Phase 2: The Right Approach

Structured error propagation: each agent returns error context when it fails, and the coordinator makes informed decisions about how to proceed.

In [ ]:
# Same failures, but with structured error reporting
db_result = simulate_database_agent(query, fail_mode="timeout_structured")
web_result = simulate_web_agent(query, fail_mode="rate_limited_structured")
file_result = simulate_file_agent(query)  # succeeds

all_results = {
    "database": db_result,
    "web_search": web_result,
    "file_reader": file_result
}

structured_coordinator_prompt = f"""You are a research coordinator. You sent a query to three data sources.
Here are the results (some may contain errors):

{json.dumps(all_results, indent=2)}

Query: {query}

INSTRUCTIONS:
1. Check each source's status. If status is "error", note what failed and why.
2. Distinguish between:
   - ACCESS FAILURE (timeout, rate limit, permission denied) = "we couldn't look"
   - EMPTY RESULTS with status "success" = "we looked and found nothing"
3. Synthesize an answer from successful sources only.
4. EXPLICITLY report which sources failed and why.
5. Rate confidence based on what data you ACTUALLY have, not what you expected.

Respond as JSON: {{
  "answer": "...",
  "confidence": "high/medium/low",
  "sources_used": [...],
  "sources_failed": [{{"source": "...", "error": "...", "suggested_action": "..."}}],
  "data_gaps": ["what information might be missing"]
}}"""

response = client.messages.create(
    model=MODEL,
    max_tokens=700,
    messages=[{"role": "user", "content": structured_coordinator_prompt}]
)

print("=== STRUCTURED PROPAGATION: Coordinator's Answer ===")
print(response.content[0].text)
print()
print("THE DIFFERENCE:")
print("  - Coordinator KNOWS the database timed out")
print("  - Coordinator KNOWS web search was rate-limited")
print("  - Reports LOW confidence because 2/3 sources failed")
print("  - Suggests retry actions for each failure")

---

## Phase 3: Your Turn

Build structured error propagation for a customer order pipeline with three stages.

In [ ]:
# Pipeline: inventory check -> payment processing -> shipping estimation
# Each stage can fail in different ways

PIPELINE_STAGES = [
    {
        "name": "inventory_check",
        "success_result": {"sku": "WIDGET-X", "in_stock": True, "quantity_available": 42},
        "failure_modes": {
            "timeout": {"error_type": "connection_timeout", "service": "inventory_db"},
            "empty": {"error_type": "product_not_found", "sku_queried": "WIDGET-X"}
        }
    },
    {
        "name": "payment_processing",
        "success_result": {"transaction_id": "TXN-99821", "amount": 149.99, "status": "approved"},
        "failure_modes": {
            "declined": {"error_type": "card_declined", "reason": "insufficient_funds"},
            "gateway_down": {"error_type": "service_unavailable", "service": "payment_gateway"}
        }
    },
    {
        "name": "shipping_estimation",
        "success_result": {"carrier": "FastShip", "estimate_days": 3, "cost": 12.99},
        "failure_modes": {
            "invalid_address": {"error_type": "validation_error", "field": "shipping_address"},
            "carrier_api_down": {"error_type": "service_unavailable", "service": "carrier_api"}
        }
    }
]

# Test scenario: inventory succeeds, payment gateway is down, shipping succeeds
TEST_RESULTS = [
    {"stage": "inventory_check", "status": "success", "data": PIPELINE_STAGES[0]["success_result"]},
    {"stage": "payment_processing", "status": "error", **PIPELINE_STAGES[1]["failure_modes"]["gateway_down"],
     "attempted_amount": 149.99, "partial_results": None,
     "alternatives": ["retry_after_30s", "use_backup_gateway", "queue_for_later"]},
    {"stage": "shipping_estimation", "status": "success", "data": PIPELINE_STAGES[2]["success_result"]}
]

print("Pipeline test scenario loaded.")
for r in TEST_RESULTS:
    status = "OK" if r["status"] == "success" else f"FAILED ({r.get('error_type', 'unknown')})"
    print(f"  {r['stage']}: {status}")

In [ ]:
# =============================================================
# TODO: Write a coordinator prompt that handles errors correctly
# =============================================================
#
# Requirements:
# - Must distinguish access failures from legitimate empty results
# - Must report which stages failed and suggest recovery actions
# - Must NOT proceed as if all stages succeeded
# - Must output: order_status, completed_stages, failed_stages, recovery_plan
#
# Think about:
# - Can the order proceed without payment? (No)
# - Should the coordinator retry automatically or escalate?
# - What information does the recovery plan need?

YOUR_COORDINATOR_PROMPT = f"""You are an order processing coordinator.

# TODO: Write instructions that handle the pipeline results correctly.
# Distinguish access failures from data-not-found.
# Report what succeeded, what failed, and what to do next.

Pipeline results:
{json.dumps(TEST_RESULTS, indent=2)}

Respond as JSON: {{{{
  "order_status": "...",
  "completed_stages": [...],
  "failed_stages": [...],
  "recovery_plan": "...",
  "can_proceed": true/false
}}}}"""

response = client.messages.create(
    model=MODEL,
    max_tokens=500,
    messages=[{"role": "user", "content": YOUR_COORDINATOR_PROMPT}]
)

print("=== YOUR COORDINATOR RESPONSE ===")
print(response.content[0].text)

In [ ]:
# =============================================================
# Checker: validates your solution
# =============================================================

raw = response.content[0].text.strip()
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

try:
    result = json.loads(raw)
except json.JSONDecodeError:
    print("Failed to parse JSON response. Check your prompt's output format.")
    result = {}

checks = {
    "reports_failure": result.get("can_proceed") == False or "fail" in str(result.get("order_status", "")).lower() or "pending" in str(result.get("order_status", "")).lower(),
    "identifies_payment_failure": "payment" in str(result.get("failed_stages", [])).lower(),
    "has_recovery_plan": bool(result.get("recovery_plan", "")),
    "preserves_successful_stages": "inventory" in str(result.get("completed_stages", [])).lower(),
}

print("=== YOUR SCORECARD ===")
for check, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"  {check}: {status}")

if all(checks.values()):
    print("\n  PASSED — your coordinator correctly handles pipeline errors!")
else:
    failed = [c for c, p in checks.items() if not p]
    print(f"\n  NEEDS WORK: {failed}")
    print("  Revise your coordinator prompt to handle errors explicitly.")

---

## Phase 4: Stress Test

Test your coordinator against multiple failure scenarios.

In [ ]:
# Test with different failure combinations
FAILURE_SCENARIOS = [
    {
        "name": "All succeed",
        "results": [
            {"stage": "inventory_check", "status": "success", "data": PIPELINE_STAGES[0]["success_result"]},
            {"stage": "payment_processing", "status": "success", "data": PIPELINE_STAGES[1]["success_result"]},
            {"stage": "shipping_estimation", "status": "success", "data": PIPELINE_STAGES[2]["success_result"]}
        ],
        "expected_proceed": True
    },
    {
        "name": "Inventory out of stock",
        "results": [
            {"stage": "inventory_check", "status": "error", "error_type": "product_not_found", "sku_queried": "WIDGET-X"},
            {"stage": "payment_processing", "status": "skipped", "reason": "dependent_stage_failed"},
            {"stage": "shipping_estimation", "status": "skipped", "reason": "dependent_stage_failed"}
        ],
        "expected_proceed": False
    },
    {
        "name": "Card declined",
        "results": [
            {"stage": "inventory_check", "status": "success", "data": PIPELINE_STAGES[0]["success_result"]},
            {"stage": "payment_processing", "status": "error", "error_type": "card_declined", "reason": "insufficient_funds"},
            {"stage": "shipping_estimation", "status": "skipped", "reason": "dependent_stage_failed"}
        ],
        "expected_proceed": False
    }
]

print("Testing your coordinator against multiple failure scenarios...\n")

for scenario in FAILURE_SCENARIOS:
    prompt = YOUR_COORDINATOR_PROMPT.replace(
        json.dumps(TEST_RESULTS, indent=2),
        json.dumps(scenario["results"], indent=2)
    )
    response = client.messages.create(
        model=MODEL,
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        result = json.loads(raw)
        can_proceed = result.get("can_proceed", None)
        match = can_proceed == scenario["expected_proceed"]
        tag = "CORRECT" if match else "WRONG"
        print(f"  {scenario['name']}: can_proceed={can_proceed} (expected {scenario['expected_proceed']}) — {tag}")
    except json.JSONDecodeError:
        print(f"  {scenario['name']}: Failed to parse response")

### Key Takeaways

1. **Silent suppression is worse than crashing.** Empty results as "success" produce wrong answers with no warning. Crashes at least tell you something failed.
2. **Access failures != empty results.** "We couldn't look" (timeout, permission denied) is fundamentally different from "we looked and found nothing."
3. **Structured error context enables local recovery.** error_type, attempted_query, partial_results, and alternatives give the coordinator everything needed to decide: retry, fallback, proceed with partial, or escalate.
4. **Coordinators need explicit failure-handling instructions.** Without them, even structured errors may be glossed over in synthesis.